# Exercícios — Validação e seleção de modelos

Soluções em `# @title`.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Exercício 1 — O vazamento da seleção de variáveis

Com muitas variáveis e poucas informativas, selecionar as "melhores" usando **todos** os dados antes de validar vaza o teste. Comparamos com a seleção feita **dentro** do pipeline.

In [ ]:
# @title Solução
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

# 1000 variaveis, so 5 informativas
X, y = make_classification(n_samples=200, n_features=1000, n_informative=5,
                           n_redundant=0, random_state=SEMENTE)
# ERRADO: escolhe as 20 melhores usando TODOS os dados, depois valida
X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)
ac_errado = cross_val_score(LogisticRegression(max_iter=5000), X_sel, y, cv=5).mean()
# CERTO: selecao DENTRO do pipeline (so no treino de cada dobra)
ac_certo = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20),
                                         LogisticRegression(max_iter=5000)), X, y, cv=5).mean()
print("estimativa com vazamento (selecao fora):", round(ac_errado, 3))
print("estimativa correta (selecao no pipeline):", round(ac_certo, 3))
print("o vazamento infla: a selecao espiou o teste ao escolher as variaveis.")

## Exercício 2 — Grid search com validação cruzada

In [ ]:
# @title Solução
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

w = load_wine()
grade = {"svc__C": [0.1, 1, 10], "svc__gamma": [0.001, 0.01, 0.1]}
busca = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grade, cv=5)
busca.fit(w.data, w.target)
print("melhores parametros:", busca.best_params_)
print("acuracia CV do melhor:", round(busca.best_score_, 3))
print("esse numero e otimista (max sobre muitas combinacoes); use um teste separado.")

## Exercício 3 — Curva de aprendizado

In [ ]:
# @title Solução
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve

dig = load_digits()
tam, tr, val = learning_curve(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                              dig.data, dig.target, cv=5,
                              train_sizes=np.linspace(0.1, 1.0, 6))
figura = go.Figure()
figura.add_trace(go.Scatter(x=tam, y=tr.mean(axis=1), mode="lines+markers",
                            line=dict(color=AZUL), name="treino"))
figura.add_trace(go.Scatter(x=tam, y=val.mean(axis=1), mode="lines+markers",
                            line=dict(color=VERMELHO), name="validacao"))
figura.update_layout(title="Curva de aprendizado (digits)", xaxis_title="tamanho do treino",
                     yaxis_title="acuracia", height=360, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
print("lacuna grande = overfitting (mais dados ajudam); convergir baixo = underfitting.")

## Exercício 4 — Estratificar importa

In [ ]:
# @title Solução
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# alvo desbalanceado: digito 3 vs resto
y_binario = (dig.target == 3).astype(int)
modelo = DecisionTreeClassifier(max_depth=5, random_state=SEMENTE)
for nome, cv in [("KFold", KFold(5, shuffle=True, random_state=SEMENTE)),
                 ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=SEMENTE))]:
    scores = cross_val_score(modelo, dig.data, y_binario, cv=cv)
    print(nome.ljust(16), "acuracia", round(scores.mean(), 3), "| desvio", round(scores.std(), 4))
print("o estratificado costuma ter menor desvio: dobras com proporcao de classes preservada.")